# WHOOP Data Explorer & Structure Analysis
This notebook explores the raw WHOOP JSON data to understand its structure and determine the best storage strategy.

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# Load the data
data_path = Path('../data/raw/whoop_complete_20260204_232639.json')
with open(data_path, 'r') as f:
    raw_data = json.load(f)

print(f"Top-level keys: {list(raw_data.keys())}")

## 1. Data Structure Overview

In [ ]:
# Count records in each category
print("=" * 50)
print("DATA SUMMARY")
print("=" * 50)
print(f"\nUser Info: {raw_data.get('user', {})}")
print(f"\nBody Info: {raw_data.get('body', {})}")
print(f"\nCycles:    {len(raw_data.get('cycles', []))} records")
print(f"Recovery:  {len(raw_data.get('recovery', []))} records")
print(f"Sleep:     {len(raw_data.get('sleep', []))} records")
print(f"Workouts:  {len(raw_data.get('workouts', []))} records")

## 2. Explore Each Data Type

### 2.1 Cycles (Daily Strain)

In [ ]:
# Examine a single cycle record
sample_cycle = raw_data['cycles'][0]
print("CYCLE STRUCTURE:")
print(json.dumps(sample_cycle, indent=2))

In [ ]:
# Flatten cycles to DataFrame
def flatten_cycles(cycles):
    records = []
    for c in cycles:
        record = {
            'cycle_id': c['id'],
            'start': c['start'],
            'end': c['end'],
            'timezone_offset': c['timezone_offset'],
            'score_state': c['score_state'],
            'strain': c.get('score', {}).get('strain'),
            'kilojoule': c.get('score', {}).get('kilojoule'),
            'average_heart_rate': c.get('score', {}).get('average_heart_rate'),
            'max_heart_rate': c.get('score', {}).get('max_heart_rate'),
        }
        records.append(record)
    return pd.DataFrame(records)

df_cycles = flatten_cycles(raw_data['cycles'])
df_cycles['start'] = pd.to_datetime(df_cycles['start'])
df_cycles['end'] = pd.to_datetime(df_cycles['end'])
df_cycles['date'] = df_cycles['start'].dt.date

print(f"Cycles DataFrame: {df_cycles.shape}")
df_cycles.head()

### 2.2 Recovery

In [ ]:
# Examine a single recovery record
sample_recovery = raw_data['recovery'][0]
print("RECOVERY STRUCTURE:")
print(json.dumps(sample_recovery, indent=2))

In [ ]:
# Flatten recovery to DataFrame
def flatten_recovery(recovery_list):
    records = []
    for r in recovery_list:
        score = r.get('score', {})
        record = {
            'cycle_id': r['cycle_id'],
            'sleep_id': r['sleep_id'],
            'created_at': r['created_at'],
            'score_state': r['score_state'],
            'recovery_score': score.get('recovery_score'),
            'resting_heart_rate': score.get('resting_heart_rate'),
            'hrv_rmssd_milli': score.get('hrv_rmssd_milli'),
            'spo2_percentage': score.get('spo2_percentage'),
            'skin_temp_celsius': score.get('skin_temp_celsius'),
            'user_calibrating': score.get('user_calibrating'),
        }
        records.append(record)
    return pd.DataFrame(records)

df_recovery = flatten_recovery(raw_data['recovery'])
df_recovery['created_at'] = pd.to_datetime(df_recovery['created_at'])
df_recovery['date'] = df_recovery['created_at'].dt.date

print(f"Recovery DataFrame: {df_recovery.shape}")
df_recovery.head()

### 2.3 Sleep

In [ ]:
# Examine a single sleep record
sample_sleep = raw_data['sleep'][0]
print("SLEEP STRUCTURE:")
print(json.dumps(sample_sleep, indent=2))

In [ ]:
# Flatten sleep to DataFrame
def flatten_sleep(sleep_list):
    records = []
    for s in sleep_list:
        score = s.get('score', {})
        stage = score.get('stage_summary', {})
        need = score.get('sleep_needed', {})
        
        record = {
            'sleep_id': s['id'],
            'cycle_id': s['cycle_id'],
            'start': s['start'],
            'end': s['end'],
            'timezone_offset': s['timezone_offset'],
            'nap': s['nap'],
            'score_state': s['score_state'],
            # Stage summary (convert milli to hours)
            'total_in_bed_hours': stage.get('total_in_bed_time_milli', 0) / 3600000,
            'total_awake_hours': stage.get('total_awake_time_milli', 0) / 3600000,
            'total_light_sleep_hours': stage.get('total_light_sleep_time_milli', 0) / 3600000,
            'total_sws_hours': stage.get('total_slow_wave_sleep_time_milli', 0) / 3600000,
            'total_rem_hours': stage.get('total_rem_sleep_time_milli', 0) / 3600000,
            'sleep_cycles': stage.get('sleep_cycle_count'),
            'disturbances': stage.get('disturbance_count'),
            # Sleep metrics
            'respiratory_rate': score.get('respiratory_rate'),
            'sleep_performance_pct': score.get('sleep_performance_percentage'),
            'sleep_consistency_pct': score.get('sleep_consistency_percentage'),
            'sleep_efficiency_pct': score.get('sleep_efficiency_percentage'),
            # Sleep need
            'baseline_need_hours': need.get('baseline_milli', 0) / 3600000,
            'debt_need_hours': need.get('need_from_sleep_debt_milli', 0) / 3600000,
            'strain_need_hours': need.get('need_from_recent_strain_milli', 0) / 3600000,
        }
        records.append(record)
    return pd.DataFrame(records)

df_sleep = flatten_sleep(raw_data['sleep'])
df_sleep['start'] = pd.to_datetime(df_sleep['start'])
df_sleep['end'] = pd.to_datetime(df_sleep['end'])
df_sleep['date'] = df_sleep['start'].dt.date

print(f"Sleep DataFrame: {df_sleep.shape}")
print(f"\nNaps vs Main Sleep:")
print(df_sleep['nap'].value_counts())
df_sleep.head()

### 2.4 Workouts

In [ ]:
# Examine a single workout record
sample_workout = raw_data['workouts'][0]
print("WORKOUT STRUCTURE:")
print(json.dumps(sample_workout, indent=2))

In [ ]:
# Flatten workouts to DataFrame
def flatten_workouts(workout_list):
    records = []
    for w in workout_list:
        score = w.get('score', {})
        zones = score.get('zone_durations', {})
        
        record = {
            'workout_id': w['id'],
            'start': w['start'],
            'end': w['end'],
            'timezone_offset': w['timezone_offset'],
            'sport_name': w.get('sport_name'),
            'sport_id': w.get('sport_id'),
            'score_state': w['score_state'],
            # Workout metrics
            'strain': score.get('strain'),
            'average_heart_rate': score.get('average_heart_rate'),
            'max_heart_rate': score.get('max_heart_rate'),
            'kilojoule': score.get('kilojoule'),
            'distance_meter': score.get('distance_meter'),
            'altitude_gain_meter': score.get('altitude_gain_meter'),
            'percent_recorded': score.get('percent_recorded'),
            # HR Zones (convert milli to minutes)
            'zone_0_mins': zones.get('zone_zero_milli', 0) / 60000,
            'zone_1_mins': zones.get('zone_one_milli', 0) / 60000,
            'zone_2_mins': zones.get('zone_two_milli', 0) / 60000,
            'zone_3_mins': zones.get('zone_three_milli', 0) / 60000,
            'zone_4_mins': zones.get('zone_four_milli', 0) / 60000,
            'zone_5_mins': zones.get('zone_five_milli', 0) / 60000,
        }
        records.append(record)
    return pd.DataFrame(records)

df_workouts = flatten_workouts(raw_data['workouts'])
df_workouts['start'] = pd.to_datetime(df_workouts['start'])
df_workouts['end'] = pd.to_datetime(df_workouts['end'])
df_workouts['date'] = df_workouts['start'].dt.date
df_workouts['duration_mins'] = (df_workouts['end'] - df_workouts['start']).dt.total_seconds() / 60

print(f"Workouts DataFrame: {df_workouts.shape}")
print(f"\nWorkout Types:")
print(df_workouts['sport_name'].value_counts().head(10))
df_workouts.head()

## 3. Data Quality Check

In [ ]:
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

for name, df in [('Cycles', df_cycles), ('Recovery', df_recovery), 
                 ('Sleep', df_sleep), ('Workouts', df_workouts)]:
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Date Range: {df['date'].min()} to {df['date'].max()}")
    print(f"  Missing values:")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        for col, count in missing.items():
            print(f"    - {col}: {count} ({count/len(df)*100:.1f}%)")
    else:
        print("    None")

## 4. Relationship Analysis

In [ ]:
# Check how tables relate via cycle_id
print("RELATIONSHIP ANALYSIS (via cycle_id)")
print("=" * 50)

cycles_ids = set(df_cycles['cycle_id'])
recovery_cycle_ids = set(df_recovery['cycle_id'])
sleep_cycle_ids = set(df_sleep['cycle_id'])

print(f"\nUnique cycle_ids in Cycles:   {len(cycles_ids)}")
print(f"Unique cycle_ids in Recovery: {len(recovery_cycle_ids)}")
print(f"Unique cycle_ids in Sleep:    {len(sleep_cycle_ids)}")

print(f"\nRecovery records matching Cycles: {len(recovery_cycle_ids & cycles_ids)}")
print(f"Sleep records matching Cycles:    {len(sleep_cycle_ids & cycles_ids)}")

In [ ]:
# Check sleep_id relationship
sleep_ids = set(df_sleep['sleep_id'])
recovery_sleep_ids = set(df_recovery['sleep_id'])

print("\nRELATIONSHIP via sleep_id:")
print(f"Unique sleep_ids in Sleep:    {len(sleep_ids)}")
print(f"Unique sleep_ids in Recovery: {len(recovery_sleep_ids)}")
print(f"Matching: {len(sleep_ids & recovery_sleep_ids)}")

## 5. Create Unified Daily DataFrame

In [ ]:
# Create a unified daily view by joining on cycle_id
# Start with cycles as the base
df_daily = df_cycles[['cycle_id', 'date', 'strain', 'kilojoule', 
                       'average_heart_rate', 'max_heart_rate']].copy()
df_daily.columns = ['cycle_id', 'date', 'day_strain', 'day_kilojoule', 
                    'day_avg_hr', 'day_max_hr']

# Merge recovery
recovery_cols = ['cycle_id', 'recovery_score', 'resting_heart_rate', 
                 'hrv_rmssd_milli', 'spo2_percentage', 'skin_temp_celsius']
df_daily = df_daily.merge(df_recovery[recovery_cols], on='cycle_id', how='left')

# Merge main sleep (exclude naps)
main_sleep = df_sleep[df_sleep['nap'] == False].copy()
sleep_cols = ['cycle_id', 'total_in_bed_hours', 'total_sws_hours', 'total_rem_hours',
              'sleep_performance_pct', 'sleep_efficiency_pct', 'disturbances']
df_daily = df_daily.merge(main_sleep[sleep_cols], on='cycle_id', how='left')

# Sort by date
df_daily = df_daily.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Unified Daily DataFrame: {df_daily.shape}")
df_daily.head(10)

In [ ]:
# Quick stats on the unified data
df_daily.describe()

## 6. Storage Recommendations

In [ ]:
print("""
================================================================================
STORAGE RECOMMENDATIONS
================================================================================

Based on the data structure analysis, here are my recommendations:

OPTION 1: Parquet Files (RECOMMENDED)
--------------------------------------
Best for: Analysis, ML pipelines, efficient storage
Structure:
  data/processed/
    ├── cycles.parquet      (~50KB)
    ├── recovery.parquet    (~30KB) 
    ├── sleep.parquet       (~100KB)
    ├── workouts.parquet    (~50KB)
    └── daily_unified.parquet  (~50KB)

Pros:
  - Column-oriented (fast for analytics)
  - Compressed (5-10x smaller than CSV)
  - Preserves data types
  - Fast read/write with pandas
  
OPTION 2: SQLite Database
--------------------------------------
Best for: Complex queries, relationships, web apps
Structure:
  data/processed/whoop.db
    Tables: cycles, recovery, sleep, workouts
    Views: daily_unified

Pros:
  - SQL queries
  - Enforced relationships
  - Single file
  - Good for building apps on top

OPTION 3: CSV Files (Simple)
--------------------------------------
Best for: Excel users, simple sharing
Cons: Large files, no type preservation

================================================================================
MY RECOMMENDATION: Start with Parquet for analysis, add SQLite later if needed.
================================================================================
""")

## 7. Save Processed Data

In [ ]:
# Create processed directory
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Save as Parquet
df_cycles.to_parquet(processed_dir / 'cycles.parquet', index=False)
df_recovery.to_parquet(processed_dir / 'recovery.parquet', index=False)
df_sleep.to_parquet(processed_dir / 'sleep.parquet', index=False)
df_workouts.to_parquet(processed_dir / 'workouts.parquet', index=False)
df_daily.to_parquet(processed_dir / 'daily_unified.parquet', index=False)

print("Saved Parquet files:")
for f in processed_dir.glob('*.parquet'):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

In [ ]:
# Also save as CSV for easy inspection
df_daily.to_csv(processed_dir / 'daily_unified.csv', index=False)
print(f"\nAlso saved: daily_unified.csv")

## 8. Quick Visualization Preview

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Recovery over time
ax1 = axes[0, 0]
ax1.plot(df_daily['date'], df_daily['recovery_score'], alpha=0.7)
ax1.set_title('Recovery Score Over Time')
ax1.set_ylabel('Recovery %')
ax1.tick_params(axis='x', rotation=45)

# Strain over time
ax2 = axes[0, 1]
ax2.plot(df_daily['date'], df_daily['day_strain'], alpha=0.7, color='orange')
ax2.set_title('Daily Strain Over Time')
ax2.set_ylabel('Strain')
ax2.tick_params(axis='x', rotation=45)

# HRV distribution
ax3 = axes[1, 0]
ax3.hist(df_daily['hrv_rmssd_milli'].dropna(), bins=30, alpha=0.7, color='green')
ax3.set_title('HRV Distribution')
ax3.set_xlabel('HRV (ms)')

# Recovery vs Strain scatter
ax4 = axes[1, 1]
ax4.scatter(df_daily['recovery_score'], df_daily['day_strain'], alpha=0.5)
ax4.set_title('Recovery vs Strain')
ax4.set_xlabel('Recovery %')
ax4.set_ylabel('Strain')

plt.tight_layout()
plt.savefig(processed_dir / 'quick_overview.png', dpi=100)
plt.show()

print("\nSaved: quick_overview.png")

## Next Steps

Now that you have clean, structured data, you can:

1. **Build a Recovery Prediction Model** - Use `daily_unified.parquet` to predict recovery based on sleep, strain, and other factors
2. **Time Series Analysis** - Look for patterns, seasonality, trends
3. **Feature Engineering** - Add rolling averages, lag features, etc.
4. **Dashboard** - Build a Streamlit or Plotly dashboard